# GenAI-Net (RL4CRN) Tutorial 04 — Dose–Response Matching Task

Compact tutorial for the **dose–response** task.

Goal: learn a CRN such that the output matches a target function of the input dose (e.g., Hill function).

---
## 0) Environment sanity check


In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


---
## 1) Imports

We use the standard interface layer:
- `Configurator`, `make_task`, `make_session_and_trainer`


In [ ]:
import numpy as np

from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
)


---
## 2) Notebook-local helpers


In [ ]:
def print_task_summary(task, max_preview=3):
    print("Task:", task.name)
    print("time_horizon:", task.time_horizon.shape, f"[0..{task.time_horizon[-1]}]")
    print("num scenarios:", len(task.u_list))
    if len(task.u_list) > 0:
        print(f"first {min(max_preview, len(task.u_list))} u:", task.u_list[:max_preview])
    print()

def run_smoke_reward(task, state, label=""):
    out = task.compute_reward(state)
    if isinstance(out, tuple):
        loss, info = out
    else:
        loss, info = out, {}
    print(f"[reward smoke{(' - ' + label) if label else ''}] loss={float(loss):.6g} | info_keys={list(info.keys())[:10]}")
    return out


---
## 3) Define a target function and task (standalone)


In [ ]:
def hill_function(u: float, kd=5.0, max_production=50.0, n=6.0) -> float:
    return max_production * (u**n) / (kd**n + u**n)

species_labels = ["X_1","X_2","OUT"]

task = make_task(
    kind="dose_response",
    species_labels=species_labels,
    dose_range=(0.0, 10.0, 10),
    target_fn=hill_function,
    ic=("constant", 0.1),
    weights="transient",
    t_f=50, n_t=120,
)

print_task_summary(task)


---
## 5) Full wiring + training loop (compact)

Pattern:
1. Configure `cfg`
2. Build `session, trainer`
3. Smoke-test reward on the template
4. Run a short training
5. Inspect the best


In [ ]:
cfg = Configurator.preset("fast")

# ---- Task ----
cfg.task.kind = "dose_response"
cfg.task.n_inputs = 1
cfg.task.t_f = 50.0
cfg.task.N_t = 120
cfg.task.ic_value = 0.1
cfg.task.weights = "transient"
cfg.task.target_fn = lambda u: hill_function(u, kd=5.0, max_production=50.0, n=6.0)
cfg.task.dose_range = (0.0, 10.0, 10)

# ---- Train ----
cfg.train.max_added_reactions = 4
cfg.train.epochs = 8
cfg.train.render_every = 2
cfg.train.seed = 3

session, trainer = make_session_and_trainer(cfg, device="auto")
print_task_summary(session.task)
run_smoke_reward(session.task, session.crn_template, label="template")

# Training can be slower depending on solver settings; keep compact by default
trainer.run(epochs=cfg.train.epochs, checkpoint_path=None)
trainer.inspect_best(plot=True)


---
## 6) Customize

Try:
- swapping in your own `target_fn` (measured dose-response curve fit)
- changing the sampling density `dose_range=(min,max,num)`
- switching reward emphasis via `weights`
